In [ ]:
!pip -q install catboost

In [ ]:
!pip -q install lightgbm

In [ ]:
!pip -q install tabpfn

In [ ]:
!pip install "tabpfn-extensions[hpo]

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [ ]:
!pip install huggingface_hub

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [ ]:
#To use TabPFN model
from huggingface_hub import login

token = input("Enter your Hugging Face token: ")
login(token)

In [ ]:
!pip install optuna wandb
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, RandomizedSearchCV, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, make_scorer

from lightgbm import LGBMRegressor

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print(train.shape)
train.head()

y = train["SalePrice"]
X = train.drop("SalePrice", axis=1)

X = X.drop("Id", axis=1)
test_ids = test["Id"]
test = test.drop("Id", axis=1)
#The competition uses log RMSE, so we log-transform.
y = np.log1p(y)

#Outliar removal
mask = ~((X["GrLivArea"] > 4000) & (np.expm1(y) < 300000))

X = X.loc[mask].reset_index(drop=True)
y = y.loc[mask].reset_index(drop=True)

# Feature Engineering
for df in [X, test]:
    df["TotalSF"] = df["TotalBsmtSF"].fillna(0) + df["1stFlrSF"].fillna(0) + df["2ndFlrSF"].fillna(0)
    df["TotalBathrooms"] = (
        df["FullBath"].fillna(0)
        + 0.5 * df["HalfBath"].fillna(0)
        + df["BsmtFullBath"].fillna(0)
        + 0.5 * df["BsmtHalfBath"].fillna(0)
    )
    df["HouseAge"] = (df["YrSold"] - df["YearBuilt"]).clip(lower=0)
    df["RemodAge"] = (df["YrSold"] - df["YearRemodAdd"]).clip(lower=0)

for df in [X, test]:
    df["MSSubClass"] = df["MSSubClass"].astype(str)

for df in [X, test]:

    df["TotalPorchSF"] = (
        df["OpenPorchSF"].fillna(0)
        + df["3SsnPorch"].fillna(0)
        + df["EnclosedPorch"].fillna(0)
        + df["ScreenPorch"].fillna(0)
        + df["WoodDeckSF"].fillna(0)
    )

    df["TotalHouseSF"] = (
        df["TotalBsmtSF"].fillna(0)
        + df["1stFlrSF"].fillna(0)
        + df["2ndFlrSF"].fillna(0)
        + df["GarageArea"].fillna(0)
    )

    df["TotalRooms"] = (
        df["TotRmsAbvGrd"].fillna(0)
        + df["FullBath"].fillna(0)
        + df["HalfBath"].fillna(0)
    )

    df["HasBasement"] = (df["TotalBsmtSF"] > 0).astype(int)
    df["HasGarage"] = (df["GarageArea"] > 0).astype(int)
    df["Has2ndFloor"] = (df["2ndFlrSF"] > 0).astype(int)
    df["HasFireplace"] = (df["Fireplaces"] > 0).astype(int)

    df["AgeAtSale"] = df["YrSold"] - df["YearBuilt"]
    df["IsRemodeled"] = (df["YearRemodAdd"] != df["YearBuilt"]).astype(int)

    df["OverallQual_TotalSF"] = df["OverallQual"] * df["TotalSF"]
    df["OverallQual2"] = df["OverallQual"] ** 2
#Handle skewness
numeric_feats = X.select_dtypes(include=["int64", "float64"]).columns

skewness = X[numeric_feats].skew(numeric_only=True)
skewed_features = skewness[skewness > 0.75].index

# keep only columns that are nonnegative in both train and test
safe_skewed_features = [
    col for col in skewed_features
    if (X[col].min() >= 0) and (test[col].min() >= 0)
]

X[safe_skewed_features] = np.log1p(X[safe_skewed_features])
test[safe_skewed_features] = np.log1p(test[safe_skewed_features])

num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns
print(len(num_cols))
print(len(cat_cols))

#median imputation is chosen as it's more robust to outliers
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

kf = KFold(n_splits=5, shuffle=True, random_state=42)

rmse_scorer = make_scorer(
    lambda yt, yp: np.sqrt(mean_squared_error(yt, yp)),
    greater_is_better=False
)



Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
(1460, 81)
45
44


In [ ]:
#LGBM Optuna tuning

lgbm = LGBMRegressor(
    objective="regression",
    random_state=42,
    n_jobs=1
)

pipe_lgbm = Pipeline([
    ("preprocess", preprocess),
    ("model", lgbm)
])

rmse_scorer = make_scorer(
    lambda yt, yp: np.sqrt(mean_squared_error(yt, yp)),
    greater_is_better=False
)

kf = KFold(n_splits=10, shuffle=True, random_state=42)


import optuna
import wandb
from sklearn.model_selection import cross_val_score

project_name = "kaggle-houseprices-hpo"
wandb.init(project=project_name, name="optuna-lgbm", reinit=True)

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 800, 3500),
        "learning_rate": trial.suggest_float("learning_rate", 0.002, 0.08, log=True),

        "num_leaves": trial.suggest_int("num_leaves", 16, 300),
        "max_depth": trial.suggest_int("max_depth", -1, 14),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),

        "subsample": trial.suggest_float("subsample", 0.2, 1.0),
        "subsample_freq": trial.suggest_int("subsample_freq", 0, 8),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),

        "min_split_gain": trial.suggest_float("min_split_gain", 1e-9, 1e-2, log=True),
        "max_bin": trial.suggest_int("max_bin", 64, 400),

        "reg_alpha": trial.suggest_float("reg_alpha", 1e-6, 20.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-6, 20.0, log=True),
    }


    pipe_lgbm.set_params(
        model__n_jobs=1,
        model__verbosity=-1,
        **{f"model__{k}": v for k, v in params.items()}
    )

    scores = cross_val_score(pipe_lgbm, X, y, cv=kf, scoring=rmse_scorer, n_jobs=-1)
    rmse = -scores.mean()

    wandb.log({"trial": trial.number, "cv_rmse": rmse, **params})
    return rmse

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20, show_progress_bar=True)

wandb.summary["best_cv_rmse"] = study.best_value
wandb.summary["best_params"] = study.best_params
wandb.finish()

print("Best RMSE:", study.best_value)
print("Best params:", study.best_params)

[I 2026-03-15 06:38:59,676] A new study created in memory with name: no-name-8dca37b9-6f10-48ff-b79c-472a8cbe5284


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-03-15 06:39:11,527] Trial 0 finished with value: 0.12638076786263996 and parameters: {'n_estimators': 2419, 'learning_rate': 0.017045054046888095, 'num_leaves': 152, 'max_depth': 3, 'min_child_samples': 91, 'subsample': 0.9242011586843932, 'subsample_freq': 2, 'colsample_bytree': 0.9083175903275871, 'min_split_gain': 0.00024581111925161755, 'max_bin': 82, 'reg_alpha': 0.3901213162781651, 'reg_lambda': 1.3920941139065095}. Best is trial 0 with value: 0.12638076786263996.
[I 2026-03-15 06:39:20,846] Trial 1 finished with value: 0.12707067034535593 and parameters: {'n_estimators': 2728, 'learning_rate': 0.01756667980620516, 'num_leaves': 294, 'max_depth': 7, 'min_child_samples': 63, 'subsample': 0.5804089634853886, 'subsample_freq': 7, 'colsample_bytree': 0.9557640984009912, 'min_split_gain': 0.0035851249388579648, 'max_bin': 112, 'reg_alpha': 0.09158549506766961, 'reg_lambda': 1.9960846575201112e-06}. Best is trial 0 with value: 0.12638076786263996.
[I 2026-03-15 06:39:25,854] Tr

colsample_bytree,▇█▂▂█▆▇▄▇▇▁▁▁▃▄▅▄▆▃▂
cv_rmse,▂▂▁▂█▁▂▂▃▂▁▁▂▁▁▁▂▁▁▃
learning_rate,▂▂▁▁▁▂▂▁▁▄▇██▄▁▁▁▂▁▁
max_bin,▁▂▃▂▅▄▃▄▃▄██▇▆▆▆▆▃▁▁
max_depth,▃▅▅▇▁▆▆▁▁▃███▅▃▃▃▅▃▃
min_child_samples,█▆▂▂▃▄▆██▆▁▁▃▂▁▃▅▂▁▄
min_split_gain,▁▅▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁
n_estimators,▅▆▅▁▁▇▇▄▄▁▃▃▃▅▃▄▅█▃▂
num_leaves,▄█▃▆▅▃▂█▆▆▁▁▃▂▃▄▄▂▅▅
reg_alpha,▂▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+4,...


Best RMSE: 0.11566774773546282
Best params: {'n_estimators': 2286, 'learning_rate': 0.004194778200850804, 'num_leaves': 95, 'max_depth': 7, 'min_child_samples': 14, 'subsample': 0.4060385871847091, 'subsample_freq': 1, 'colsample_bytree': 0.3900750697689478, 'min_split_gain': 6.3033679097529095e-09, 'max_bin': 145, 'reg_alpha': 1.543747160153332e-05, 'reg_lambda': 1.9646988710769282}


In [ ]:
# LGBM submission using Optuna-best params
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline


final_lgbm = LGBMRegressor(
    objective="regression",
    random_state=42,
    n_jobs=-1,
    verbosity=-1,

    n_estimators=2286,
    learning_rate= 0.004194778200850804,
    num_leaves= 95,
    max_depth= 7,
    min_child_samples= 14,
    subsample= 0.4060385871847091,
    subsample_freq= 1,
    colsample_bytree= 0.3900750697689478,
    min_split_gain= 6.3033679097529095e-09,
    max_bin= 145,
    reg_alpha= 1.543747160153332e-05,
    reg_lambda= 1.9646988710769282
)

final_model_lgbm = Pipeline([
    ("preprocess", preprocess),
    ("model", final_lgbm)
])

# Fit on all training data
final_model_lgbm.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

In [ ]:
#Catboost optuna tuning

kf = KFold(n_splits=10, shuffle=True, random_state=42)

rmse_scorer = make_scorer(
    lambda yt, yp: np.sqrt(mean_squared_error(yt, yp)),
    greater_is_better=False
)
#Optuna tuning for Catboost:
import optuna
import wandb
from sklearn.model_selection import cross_val_score
from catboost import CatBoostRegressor
from sklearn.pipeline import Pipeline


# kf = KFold(n_splits=10, shuffle=True, random_state=42)


# Base CatBoost pipeline
cat_model = CatBoostRegressor(
    loss_function="RMSE",
    random_state=42,
    verbose=0
)

pipe_cat = Pipeline([
    ("preprocess", preprocess),
    ("model", cat_model)
])

project_name = "kaggle-houseprices-hpo"
wandb.init(project=project_name, name="optuna-catboost", reinit=False)

def objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 600, 2500),
        "learning_rate": trial.suggest_float("learning_rate", 0.002, 0.02, log=True),
        "depth": trial.suggest_int("depth", 4, 7),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 20.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 10.0),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 30),
        "border_count": trial.suggest_int("border_count", 32, 200),
    }

    pipe_cat.set_params(
        model__thread_count=1,
        model__verbose=0,
        **{f"model__{k}": v for k, v in params.items()}
    )

    scores = cross_val_score(pipe_cat, X, y, cv=kf, scoring=rmse_scorer, n_jobs=1)
    rmse = -scores.mean()

    wandb.log({"trial": trial.number, "cv_rmse": rmse, **params})
    return rmse

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20, show_progress_bar=True)

wandb.summary["best_cv_rmse"] = study.best_value
wandb.summary["best_params"] = study.best_params
wandb.finish()

print("Best RMSE:", study.best_value)
print("Best params:", study.best_params)

In [ ]:
#Optuna catboost model submission

# Final Kaggle submission using Optuna-best CatBoost params
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.pipeline import Pipeline

final_cat = CatBoostRegressor(
    loss_function="RMSE",
    random_state=42,
    verbose=0,
    thread_count=-1,

    iterations=819,
    learning_rate=0.019912604411637558,
    depth=7,
    l2_leaf_reg=0.5395057092322707,
    random_strength=0.4183800857939707,
    bagging_temperature=2.6845884800717412,
    min_data_in_leaf=21,
    border_count=131
)


final_model_cat = Pipeline([
    ("preprocess", preprocess),
    ("model", final_cat)
])

# Fit on all training data
final_model_cat.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

In [ ]:
#XGBoost

xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=1,
    tree_method="hist"
)

pipe_xgb = Pipeline([
    ("preprocess", preprocess),
    ("model", xgb)
])
#XGBoost Optuna Tuning

import optuna
import numpy as np
from sklearn.model_selection import cross_val_score

project_name = "kaggle-houseprices-hpo"
wandb.init(project=project_name, name="optuna-xgb", reinit=True)

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 500, 3000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 7),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 12),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "gamma": trial.suggest_float("gamma", 1e-9, 1e-2, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-6, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 30.0, log=True),
    }

    pipe_xgb.set_params(
        model__n_jobs=1, model__tree_method="hist",
        **{f"model__{k}": v for k, v in params.items()}
    )

    scores = cross_val_score(pipe_xgb, X, y, cv=kf, scoring=rmse_scorer, n_jobs=-1)
    rmse = -scores.mean()

    # log each trial
    wandb.log({"trial": trial.number, "cv_rmse": rmse, **params})
    return rmse

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20, show_progress_bar=True)

wandb.summary["best_cv_rmse"] = study.best_value
wandb.summary["best_params"] = study.best_params
wandb.finish()

print("Best RMSE:", study.best_value)
print("Best params:", study.best_params)

[I 2026-03-15 05:33:25,884] A new study created in memory with name: no-name-55044b3d-3541-44c8-a47f-fc1ad4e1e245


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-03-15 05:33:38,416] Trial 0 finished with value: 0.11732689965520293 and parameters: {'n_estimators': 1081, 'learning_rate': 0.013902396986245527, 'max_depth': 4, 'min_child_weight': 10, 'subsample': 0.8048340724219748, 'colsample_bytree': 0.9236860730597668, 'gamma': 6.404371522851788e-07, 'reg_alpha': 2.4749018436949246e-05, 'reg_lambda': 0.001744056371030429}. Best is trial 0 with value: 0.11732689965520293.
[I 2026-03-15 05:34:01,284] Trial 1 finished with value: 0.11779355506517838 and parameters: {'n_estimators': 2594, 'learning_rate': 0.035243075248510065, 'max_depth': 6, 'min_child_weight': 8, 'subsample': 0.8049960900055957, 'colsample_bytree': 0.725767599281847, 'gamma': 1.8540800597020856e-05, 'reg_alpha': 0.427939930860223, 'reg_lambda': 2.7992751583826028}. Best is trial 0 with value: 0.11732689965520293.
[I 2026-03-15 05:34:15,344] Trial 2 finished with value: 0.11756652247185731 and parameters: {'n_estimators': 1301, 'learning_rate': 0.010981935234572257, 'max_de

colsample_bytree,▆▂▇▃▂█▃▇▂█▅▅▅▄▃▆▄▅▄▁
cv_rmse,▄▄▄█▇▅▅▄▁▇▁▁▁▁▃▄▂▂▁▁
gamma,▁▁▁█▁▃▄▁▁▁▁▁▁▁▁▁▁▁▁▁
learning_rate,▁▃▁█▁▂▄▇▄▅▂▂▂▂▁▂▃▂▃▂
max_depth,▄▇█▇█▅█▇▂▇▁▁▁▁▂▂▁▄▂▁
min_child_weight,▇▅▃▂▂▇▅▄▄▂▄▄▃▃█▃▁▅▂▅
n_estimators,▃▇▃▆▂▄▃▁▆▄▆▆▅███▅▇█▅
reg_alpha,▁▅▁█▁▁▁▁▁▆▁▁▁▁▁▁▁▁▁▁
reg_lambda,▁▂▆▁█▁▁▂▆▁▁▁▁▁▁▁▁▁▁▁
subsample,▄▄▃▆▇▆▇▃▆▆▁▁▁▁▂█▂▅▂▅
+1,...


Best RMSE: 0.11302929769895183
Best params: {'n_estimators': 2905, 'learning_rate': 0.01808305347457831, 'max_depth': 2, 'min_child_weight': 4, 'subsample': 0.711720990274584, 'colsample_bytree': 0.8398044003590183, 'gamma': 1.3118657281271936e-09, 'reg_alpha': 1.5939625399645739e-06, 'reg_lambda': 0.031743092091746736}


In [ ]:
# XGBoost Final model using Optuna best params

from xgboost import XGBRegressor
import numpy as np
import pandas as pd

final_xgb = XGBRegressor(
    n_estimators=2905,
    learning_rate=0.01808305347457831,
    max_depth=2,
    min_child_weight=4,
    subsample=0.711720990274584,
    colsample_bytree=0.8398044003590183,
    gamma=1.3118657281271936e-09,
    reg_alpha=1.5939625399645739e-06,
    reg_lambda=0.031743092091746736,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42
)

# Full pipeline
xgb_optuna_model = Pipeline([
    ("preprocess", preprocess),
    ("model", final_xgb)
])

# Train on ALL training data
xgb_optuna_model.fit(X, y)

# Predict
preds = xgb_optuna_model.predict(test)

# Reverse log transform
preds = np.expm1(preds)

# Create submission
submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": preds
})

# Save file
submission.to_csv("submission_xgb_optuna_n20.csv", index=False)

print("Submission file created!")

Submission file created!


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_pipeline_linear = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline_linear = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess_linear = ColumnTransformer([
    ("num", num_pipeline_linear, num_cols),
    ("cat", cat_pipeline_linear, cat_cols)
])

In [ ]:
# Lasso Optuna tuning

import optuna
import wandb
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

project_name = "kaggle-houseprices-hpo"
wandb.init(project=project_name, name="optuna-lasso", reinit=True)

pipe_lasso = Pipeline([
    ("preprocess", preprocess_linear),
    ("model", Lasso(random_state=42))
])

def objective(trial):
    params = {
        "alpha": trial.suggest_float("alpha", 1e-5, 1e-2, log=True),
        "max_iter": trial.suggest_int("max_iter", 5000, 30000),
        "tol": trial.suggest_float("tol", 1e-5, 1e-2, log=True),
        "selection": trial.suggest_categorical("selection", ["cyclic", "random"])
    }

    pipe_lasso.set_params(
        **{f"model__{k}": v for k, v in params.items()}
    )

    scores = cross_val_score(
        pipe_lasso,
        X,
        y,
        cv=kf,
        scoring=rmse_scorer,
        n_jobs=-1
    )

    rmse = -scores.mean()

    wandb.log({"trial": trial.number, "cv_rmse": rmse, **params})
    return rmse

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20, show_progress_bar=True)

wandb.summary["best_cv_rmse"] = study.best_value
wandb.summary["best_params"] = study.best_params
wandb.finish()

print("Best RMSE:", study.best_value)
print("Best params:", study.best_params)

[I 2026-03-17 00:01:42,785] A new study created in memory with name: no-name-3c1182b9-90c0-4025-8959-40e60f0f3d0c


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-03-17 00:02:00,698] Trial 0 finished with value: 0.1187971136038932 and parameters: {'alpha': 3.852193229858732e-05, 'max_iter': 17374, 'tol': 0.0011317445972952075, 'selection': 'random'}. Best is trial 0 with value: 0.1187971136038932.
[I 2026-03-17 00:02:25,041] Trial 1 finished with value: 0.11489617246201686 and parameters: {'alpha': 7.204880385930814e-05, 'max_iter': 21701, 'tol': 8.963744762341591e-05, 'selection': 'cyclic'}. Best is trial 1 with value: 0.11489617246201686.
[I 2026-03-17 00:02:26,741] Trial 2 finished with value: 0.10918060262641974 and parameters: {'alpha': 0.00019654028781003046, 'max_iter': 14515, 'tol': 0.0006340455811076241, 'selection': 'cyclic'}. Best is trial 2 with value: 0.10918060262641974.
[I 2026-03-17 00:02:52,970] Trial 3 finished with value: 0.11767163933928908 and parameters: {'alpha': 4.676229191660499e-05, 'max_iter': 13491, 'tol': 1.2613554580864556e-05, 'selection': 'cyclic'}. Best is trial 2 with value: 0.10918060262641974.
[I 2026-

alpha,▁▁▁▁▁▁▅▃▁▄▂▂▂▁▁▁▃▁▂█
cv_rmse,▅▄▂▅▅▂▇▅▁▆▁▁▂▁▁▇▄▁▃█
max_iter,▄▆▃▃▁▇▆▆▆▃██▇▅▅▅▅▂▁▂
tol,▂▁▂▁▁▃▂▂▁▂██▁▆▁▃▁▁▅▁
trial,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
alpha,0.00929
best_cv_rmse,0.10717
cv_rmse,0.12637
max_iter,9602
selection,random
tol,3e-05


Best RMSE: 0.10717028948146845
Best params: {'alpha': 0.0006045138127381142, 'max_iter': 19761, 'tol': 0.00420602257962069, 'selection': 'random'}


In [ ]:
# Lasso Submission file:

import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline

# Final tuned Lasso model
final_lasso = Lasso(
    alpha=0.0006045138127381142,
    max_iter=19761,
    tol=0.00420602257962069,
    selection="random",
    random_state=42
)

final_model_lasso = Pipeline([
    ("preprocess", preprocess_linear),
    ("model", final_lasso)
])

# Train on ALL training data
final_model_lasso.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

In [ ]:
#Elastic Net Optuna tuning
# ElasticNet Optuna tuning

import optuna
import wandb
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

project_name = "kaggle-houseprices-hpo"
wandb.init(project=project_name, name="optuna-elasticnet", reinit=True)

pipe_enet = Pipeline([
    ("preprocess", preprocess_linear),
    ("model", ElasticNet(random_state=42))
])

def objective(trial):
    params = {
        "alpha": trial.suggest_float("alpha", 5e-6, 1e-2, log=True),
        "l1_ratio": trial.suggest_float("l1_ratio", 0.1, 0.99),
        "max_iter": trial.suggest_int("max_iter", 15000, 40000),
        "tol": trial.suggest_float("tol", 5e-6, 1e-2, log=True),
        "selection": trial.suggest_categorical("selection", ["cyclic", "random"])
    }

    pipe_enet.set_params(
        **{f"model__{k}": v for k, v in params.items()}
    )

    scores = cross_val_score(
        pipe_enet,
        X,
        y,
        cv=kf,
        scoring=rmse_scorer,
        n_jobs=-1
    )

    rmse = -scores.mean()

    wandb.log({"trial": trial.number, "cv_rmse": rmse, **params})
    return rmse

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=80, show_progress_bar=True)

wandb.summary["best_cv_rmse"] = study.best_value
wandb.summary["best_params"] = study.best_params
wandb.finish()

print("Best RMSE:", study.best_value)
print("Best params:", study.best_params)


[I 2026-03-17 00:38:50,879] A new study created in memory with name: no-name-1919272b-b6a7-407a-aafe-3ad482ee0c8d


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-03-17 00:38:52,534] Trial 0 finished with value: 0.10944575630928173 and parameters: {'alpha': 0.000801799317066695, 'l1_ratio': 0.16133906768045922, 'max_iter': 30853, 'tol': 0.000943063733086637, 'selection': 'random'}. Best is trial 0 with value: 0.10944575630928173.
[I 2026-03-17 00:39:56,023] Trial 1 finished with value: 0.12509846178618755 and parameters: {'alpha': 8.114050315823347e-06, 'l1_ratio': 0.676255286994907, 'max_iter': 28965, 'tol': 0.0012382495532564871, 'selection': 'random'}. Best is trial 0 with value: 0.10944575630928173.
[I 2026-03-17 00:39:57,562] Trial 2 finished with value: 0.10810222304455708 and parameters: {'alpha': 0.00058716300118054, 'l1_ratio': 0.47131166964823323, 'max_iter': 19188, 'tol': 7.311789771049151e-05, 'selection': 'random'}. Best is trial 2 with value: 0.10810222304455708.
[I 2026-03-17 00:39:59,058] Trial 3 finished with value: 0.1235135743693156 and parameters: {'alpha': 0.00617131956672814, 'l1_ratio': 0.8291124728226157, 'max_ite

alpha,▁▅▁▁▁▁▃▁▁▃█▂▃▁▁▂▄▂▁▁▂▁▄▁▁▂▂▃▁▁▁▁▁▁▁▁▃▁▁▁
cv_rmse,█▁█▂▅▄▄▂▆▆▅▁▂▁▁▁▅▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▄▇▁▂▁▁▂
l1_ratio,▁▇▄▁▃▃▅▃▆▄▆▆▇▅▃▆▆▆▆▅▅▇█▇▆▇▇▆▅▆▅▆█▇██▇▄██
max_iter,▅▅█▅▅▇▆▁▆▇▃▂▃▃▃▁▁▂▂▂▃▂▂▂▃▂▂▄▃▄▅▅▃▄▃▄▄▄▅▄
tol,▂▁▁▁█▂▂▄▁▁▁▂▂▄▂▁▁▁▂▂▁▁▁▁▁▁▂▂▂▂▁▆▂▁▂▃▂▂▂▂
trial,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
alpha,0.00129
best_cv_rmse,0.10723
cv_rmse,0.10999
l1_ratio,0.93667
max_iter,27814


Best RMSE: 0.10722666633568763
Best params: {'alpha': 0.0006054138062354034, 'l1_ratio': 0.9568288981433803, 'max_iter': 25082, 'tol': 0.0009033626634930469, 'selection': 'random'}


In [ ]:
#Elastic Net Submission

from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd

final_enet = ElasticNet(
    alpha=0.0007263034643451646,
    l1_ratio=0.4737833322495446,
    max_iter=18158,
    tol=0.001170172736678686,
    selection="cyclic",
    random_state=42
)

# Full pipeline
enet_model = Pipeline([
    ("preprocess", preprocess_linear),
    ("model", final_enet)
])

# Train on ALL training data
enet_model.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

In [ ]:
!pip install -U scikit-learn

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.1 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.1 MB 2.6 MB/s eta 0:00:03
   ------- -------------------------------- 1.6/8.1 MB 3.1 MB/s eta 0:00:03
   ------------ --------------------------- 2.6/8.1 MB 3.7 MB/s eta 0:00:02
   ------------------ --------------------- 3.7/8.1 MB 4.0 MB/s eta 0:00:02
   ----------------------- ---------------- 4.7/8.1 MB 4.3 MB/s eta 0:00:01
   ---------------------------- ----------- 5.8/8.1 MB 4.3 MB/s eta 0:00:01
   -------------------------------- ------- 6.6/8.1 MB 4.3 MB/s eta 0:00:01
   ------------------------------------- -- 7.6/8.1 MB 4.4 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 4.3 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llmebench 1.1.0 requires datasets==2.14.6, but you have datasets 3.6.0 which is incompatible.
llmebench 1.1.0 requires scikit-learn==1.2.2, but you have scikit-learn 1.8.0 which is incompatible.
tabpfn-extensions 0.2.2 requires scikit-learn<1.7,>=1.6.0, but you have scikit-learn 1.8.0 which is incompatible.


In [ ]:
#KRR optuna tuning

import optuna
import wandb
from sklearn.kernel_ridge import KernelRidge
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

project_name = "kaggle-houseprices-hpo"
wandb.init(project=project_name, name="optuna-kernelridge", reinit=True)

pipe_krr = Pipeline([
    ("preprocess", preprocess_linear),
    ("model", KernelRidge())
])

def objective(trial):

    kernel = trial.suggest_categorical(
        "kernel", ["linear", "rbf", "polynomial"]
    )

    params = {
        "alpha": trial.suggest_float("alpha", 1e-5, 1e1, log=True),
        "kernel": kernel
    }

    if kernel in ["rbf", "polynomial"]:
        params["gamma"] = trial.suggest_float("gamma", 1e-4, 1, log=True)

    if kernel == "polynomial":
        params["degree"] = trial.suggest_int("degree", 2, 5)
        params["coef0"] = trial.suggest_float("coef0", 0, 2)

    pipe_krr.set_params(
        **{f"model__{k}": v for k, v in params.items()}
    )

    scores = cross_val_score(
        pipe_krr,
        X,
        y,
        cv=kf,
        scoring=rmse_scorer,
        n_jobs=-1
    )

    rmse = -scores.mean()

    wandb.log({"trial": trial.number, "cv_rmse": rmse, **params})
    return rmse


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50, show_progress_bar=True)

wandb.summary["best_cv_rmse"] = study.best_value
wandb.summary["best_params"] = study.best_params
wandb.finish()

print("Best RMSE:", study.best_value)
print("Best params:", study.best_params)

[I 2026-03-17 01:16:13,050] A new study created in memory with name: no-name-2a6d3856-58c9-409a-99d1-e11ac7abf4d7


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-03-17 01:16:14,403] Trial 0 finished with value: 0.7517693213283423 and parameters: {'kernel': 'polynomial', 'alpha': 0.037840567852569576, 'gamma': 0.00013756638362982, 'degree': 3, 'coef0': 0.07313240074838467}. Best is trial 0 with value: 0.7517693213283423.
[I 2026-03-17 01:16:15,637] Trial 1 finished with value: 0.20718755361002242 and parameters: {'kernel': 'polynomial', 'alpha': 0.2593918130400687, 'gamma': 0.04503599880774827, 'degree': 2, 'coef0': 0.8248920911102327}. Best is trial 1 with value: 0.20718755361002242.
[I 2026-03-17 01:16:16,959] Trial 2 finished with value: 0.12874513604492568 and parameters: {'kernel': 'polynomial', 'alpha': 0.0016878677486525933, 'gamma': 0.0029867132334214083, 'degree': 3, 'coef0': 0.7709810679853675}. Best is trial 2 with value: 0.12874513604492568.
[I 2026-03-17 01:16:18,303] Trial 3 finished with value: 0.15186512355179735 and parameters: {'kernel': 'polynomial', 'alpha': 0.0075188294404627916, 'gamma': 0.008607611101248152, 'degre

alpha,▁▁▁▁▁▁▁▁▄▁▁▁▁▂▇▁▁▂▂▁▁▁█▁▁▁▁▃▁▂▃▁▁▁▁▁▁▁▁▁
coef0,▁▄▄▆█▃▂▅██████▇▇▇▆▇
cv_rmse,▁▁▁▁▂▁▇▁▁▁▁▁▁▁▁▅▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁
degree,▃▁▃█▆▁▆▃██████████▆
gamma,▁▁▁▁▁▁▁▂▁▁█▂▆▁▁▁▁▁▁▁▁▁▁▁
trial,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
alpha,0.00066
best_cv_rmse,0.10818
coef0,1.59224
cv_rmse,0.11915
degree,4


Best RMSE: 0.10817590069081036
Best params: {'kernel': 'polynomial', 'alpha': 0.5160906833911447, 'gamma': 0.0004046473802349052, 'degree': 5, 'coef0': 1.9860000856141466}


In [ ]:
#KRR Submission file

# KernelRidge Final model using Optuna best params

from sklearn.kernel_ridge import KernelRidge
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd

final_krr = KernelRidge(
    alpha=0.5160906833911447,          # replace with study.best_params["alpha"]
    kernel="polynomial",       # replace with study.best_params["kernel"]
    gamma=0.0004046473802349052,        # replace with study.best_params["gamma"] if used
    degree = 5,
    coef0 = 1.9860000856141466
)

# Full pipeline
krr_model = Pipeline([
    ("preprocess", preprocess_linear),
    ("model", final_krr)
])

# Train on ALL training data
krr_model.fit(X, y)

# Predict
preds = krr_model.predict(test)

In [ ]:
# TabPFN Submission

from tabpfn import TabPFNRegressor
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd

final_tabpfn = TabPFNRegressor(
    n_estimators=50,
    random_state=42,
    device = "cuda:0"
)

tabpfn_model = Pipeline([
    ("preprocess", preprocess_linear),
    ("model", final_tabpfn)
])

tabpfn_model.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

In [ ]:
#Merging Optuna XGBoost and LightGBM Models:

pred_lgb = np.expm1(final_model_lgbm.predict(test))
pred_cat = np.expm1(final_model_cat.predict(test))
pred_enet = np.expm1(enet_model.predict(test))
pred_lasso = np.expm1(final_model_lasso.predict(test))
pred_krr = np.expm1(krr_model.predict(test))
pred_tab = np.expm1(tabpfn_model.predict(test))

pred_final = 0.1 * pred_lgb + 0.1* pred_cat + 0.1 * pred_enet + 0.1 * pred_lasso + 0.3 * pred_krr + 0.3 * pred_tab

submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": pred_final
})

submission.to_csv("submission_cat_lgbm_enet_lasso_krr_tab_2.csv", index=False)
print("Final submission file created!(50/50)")

C:\Users\yosse\anaconda3\envs\LlamaLens\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [ ]:
# TabPFN CV Evaluation

from tabpfn import TabPFNRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score
import numpy as np

# Define model
tabpfn = TabPFNRegressor(
    device="cuda:0",
)

# Full pipeline
tabpfn_model = Pipeline([
    ("preprocess", preprocess_linear),
    ("model", tabpfn)
])

# Cross-validation setup
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Evaluate CV RMSE
scores = cross_val_score(
    tabpfn_model,
    X,
    y,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=1   # important for TabPFN stability
)

rmse_scores = -scores

print("Fold RMSE:", rmse_scores)
print("Mean RMSE:", rmse_scores.mean())
print("Std RMSE:", rmse_scores.std())

Fold RMSE: [0.10605218 0.1038426  0.11331925 0.1115216  0.0919734 ]
Mean RMSE: 0.10534180367795176
Std RMSE: 0.007527231194304252
